# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations. For the iCGB21FR model, annotation of uniprot identifiers to the genes was made possible by using BLAST (work performed by Lorenzo Wormer, Karlsruhe institute of technology)

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired. Importantly, make sure to **adjust the regular expression in the cell below** in order to find the gene names associated with the organism you want to model.

In [31]:
import pandas as pd
import os
from cobra.io import load_json_model, read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

if os.path.split(os.getcwd())[1]=='i1_preprocessing':
    os.chdir('../..')
    
from PAModelpy.utils.pam_generation import merge_enzyme_complexes, get_protein_gene_mapping, _parse_gpr


DEFAULT_PROT_MW = 39959.4825 #g/mol
DEFAULT_KCAT =  13.7 #/s median value from BRENDA
DEFAULT_PROT_LENGTH = 325 # amino acids, data from E.coli (Zhang J. Protein-length distributions for the three domains of life. Trends Genet. 2000 Mar16(3):107-9.PubMed ID10689349)

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_new.xls')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_cglutamicumATCC13032_250124.xlsx')
NCBI_FILE_PATH = os.path.join('Data','Cglutanicum_phenotypes','NCBI_genes_corynglutatcc13032_250207.tsv')
GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_cgl_250124.json')
MODEL_FILE_PATH = os.path.join('Models', 'iCGB21FR_annotated.xml')

#Parameters for unused enzyme sector determination
BIOMASS_RXN_ID = 'Growth'
MAX_GROWTH_ALE = 0.96#1/h; Graf et al 2019, https://www.frontiersin.org/journals/microbiology/articles/10.3389/fmicb.2019.01648/full
UNUSED_PROTEIN_INTERCEPT = 0.37 #g_unusedprotein/g_protein; 37% (Bruggeman et al (2020)) of all the proteins which can be measured in E.coli
METABOLIC_PROTEIN_FRACTION = 0.25 #g_p/g_CDW

In [2]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'proteinAllocationModel_iCGB21FR_EnzymaticData_{formatted_date}.xlsx')
AE_OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'{formatted_date}_cgl_ActiveEnzymeInformation_GotEnzymes.xlsx')

Relatively complicated regex because of complex gene names

1. WP_\d+_\d+: Matches identifiers starting with WP_ followed by digits (\d+), an underscore, and more digits.
2. |: Alternation operator to separate different patterns.
3. lcl_NC_\d+_\d+_prot_: Matches strings starting with lcl_NC_, followed by digits, an underscore, more digits, _prot_, and then:

    - WP_\d+_\d+_\d+: Matches WP_ identifiers with an additional _ and digits at the end.
    - |\d+: Matches a plain numeric sequence following _prot_.

4. |: Alternation operator for the next part.
5. cgl\d{4}: Matches strings starting with cgl followed by exactly 4 digits.

In [3]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'(?:WP_\d+_\d+|lcl_NC_\d+_\d+_prot_(?:WP_\d+_\d+_\d+|\d+)|[cC]gl\d{4}|[cC]g\d{4})'

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [BioModels](https://www.ebi.ac.uk/biomodels/)**
- Get your model! Or use your own reaction identifiers
- In this example we will use the iCGB21FR model for [*Corynebacterium glutamicum* ATCC 13032](https://www.ebi.ac.uk/biomodels/MODEL2102050001#Overview)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `cgl` (*C. glutanicum* ATCC 13032) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be downloaded

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Go to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *C. glutanicum* we used [this](https://www.uniprot.org/uniprotkb?query=(taxonomy_id:196627)) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Download the resulting Excel file

## 1. Get information from the model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [4]:
def create_id_mapper_from_model(model):
    #make a id mapping and create a mapping dataframe
    id_mapper_df = pd.DataFrame(columns = ['rxn_id', 'kegg_id', 'Reactants','Products','EC', 'GPR'])
    for rxn in model.reactions:
        #skip transport reactions and ATP maintenance requirement
        if 'ex' in rxn.id or 'EX' in rxn.id or rxn.id == 'ATPM':
            continue
        to_append= []  
        to_append.append(rxn.id)
        #not all reactions are annotated with a KEGG ID in the model
        try: 
            to_append.append(rxn.annotation['kegg.reaction'])
        except:
            to_append.append(np.nan)
        try:
            to_append.append([reac.annotation['kegg.compound'] for reac in rxn.reactants])
        except:
            to_append.append([])
        try: 
            to_append.append([prod.annotation['kegg.compound'] for prod in rxn.products])
        except:
            to_append.append([])
        if 'ec-code' in rxn.annotation.keys():
            to_append.append(rxn.annotation['ec-code'])
            to_append.append(rxn.gene_reaction_rule)  

        else:
            to_append.append(np.nan)
            to_append.append(rxn.gene_reaction_rule)
        id_mapper_df.loc[len(id_mapper_df)] = to_append


    #remove entries without GRP relation or Ec number


    print('Number of reactions in the mapping dataframe: ',len(id_mapper_df))
    print('Number of reactions mapped to KEGG identifier: ', len(id_mapper_df.kegg_id.dropna()))
    return id_mapper_df

def create_gene_id_mapper_df(model):
    mapper_df = pd.DataFrame(columns = ['rxn_id', 'gene_id', 'uniprot_id', 'kegg_gene', 'NCBI_protein'])
    for rxn in model.reactions:
        for gene in rxn.genes:
            row = [rxn.id, gene.id]
            for dbxref in ['uniprot', 'kegg.genes','ncbiprotein']:
                if dbxref in gene.annotation:
                    row.append(gene.annotation[dbxref])
                else:
                    row.append(np.nan)
            mapper_df.loc[len(mapper_df)] = row
            
    #need to split of cgb: in front of kegg.gene id
    mapper_df['kegg_gene'] = mapper_df['kegg_gene'].apply(
    lambda gene: gene.split(':')[1] if isinstance(gene, str) and ':' in gene else gene
    )    
    print('Number of genes in the mapping dataframe: ',len(mapper_df))
    print('Number of genes mapped to KEGG identifier: ', len(mapper_df.kegg_gene.dropna()))
    print('Number of genes mapped to uniprot identifier: ', len(mapper_df.uniprot_id.dropna()))
    return mapper_df
            

print('mapping the ids from the iCGB21FR model')
model =read_sbml_model(MODEL_FILE_PATH)
id_mapper = create_id_mapper_from_model(model)
print('\nmapping gene ids')
gene_id_mapper = create_gene_id_mapper_df(model)
id_mapper

mapping the ids from the iCGB21FR model
Set parameter Username
Academic license - for non-commercial use only - expires 2025-03-06
Number of reactions in the mapping dataframe:  1272
Number of reactions mapped to KEGG identifier:  532

mapping gene ids
Number of genes in the mapping dataframe:  1659
Number of genes mapped to KEGG identifier:  1516
Number of genes mapped to uniprot identifier:  1595


,rxn_id,kegg_id,Reactants,Products,EC,GPR
0,12DGR120tipp,NaN,[],[],NaN,
1,12DGR140tipp,NaN,[],[],NaN,
2,12DGR161tipp,NaN,[],[],NaN,
3,12DGR180tipp,NaN,[],[],NaN,
4,12DGR181tipp,NaN,[],[],NaN,
...,...,...,...,...,...,...
1267,FRD7,NaN,[],[],NaN,
1268,GLYCLTt2rpp,NaN,"[C00160, C00080]","[C00160, C00080]",NaN,
1269,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1
1270,CYTB1,NaN,[],[],NaN,


### Map the NCBI gene names to KEGG gene names

In the iCGB21FR model, the gene names are by default the ncbi gene names. To map the gene-protein-reaction associations to the corresponding uniprot peptide identifiers, we need to map the model gene names to the kegg gene names.

In [5]:
gene_mapper = gene_id_mapper[['gene_id', 'kegg_gene']].set_index('gene_id').to_dict()['kegg_gene']

In [6]:
#replace the ncbi gene names in the GPR relation
def replace_ids(gpr, mapper):
    if pd.isna(gpr):  # Handle missing values
        return gpr
    for old_id, new_id in mapper.items():
        if pd.isna(new_id): continue
        gpr = gpr.replace(old_id, new_id)
    return gpr

id_mapper_gpr = id_mapper.copy()
# Apply the replacement function to the GPR column
id_mapper_gpr["GPR_old"] = id_mapper_gpr["GPR"].copy()
id_mapper_gpr["GPR"] = id_mapper_gpr["GPR"].apply(lambda x: replace_ids(x, gene_mapper))
id_mapper_gpr["GPR_old"] = id_mapper_gpr["GPR"].apply(lambda x: replace_ids(x, gene_mapper))

id_mapper_gpr

,rxn_id,kegg_id,Reactants,Products,EC,GPR,GPR_old
0,12DGR120tipp,NaN,[],[],NaN,,
1,12DGR140tipp,NaN,[],[],NaN,,
2,12DGR161tipp,NaN,[],[],NaN,,
3,12DGR180tipp,NaN,[],[],NaN,,
4,12DGR181tipp,NaN,[],[],NaN,,
...,...,...,...,...,...,...,...
1267,FRD7,NaN,[],[],NaN,,
1268,GLYCLTt2rpp,NaN,"[C00160, C00080]","[C00160, C00080]",NaN,,
1269,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1
1270,CYTB1,NaN,[],[],NaN,,


## 2. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

Parse the BIGG gene-protein-reaction associations

In [7]:
#You need to adjust this to find the geneid/locustag for your microbe
#locustag_regex =r'\b([b|s]\d{4})\b'
def extract_b_numbers(text):
    if pd.isna(text):
        return text
    return re.findall(locustag_regex, text)

id_mapper_df = id_mapper_gpr.copy()

id_mapper_df['b_number'] = id_mapper_df['GPR'].apply(extract_b_numbers)

id_mapper_df['gene_id_old'] = id_mapper_df['GPR_old'].apply(extract_b_numbers)

id_mapper_df = id_mapper_df.explode(['b_number', 'gene_id_old'], ignore_index=True)

id_mapper_df = id_mapper_df.explode('EC', ignore_index=True)

id_mapper_df = id_mapper_df.merge(
    gene_id_mapper[['uniprot_id', 'gene_id']], left_on='gene_id_old', right_on='gene_id', how = 'left'
).drop(['gene_id_old', 'GPR_old'], axis=1)

id_mapper_df

,rxn_id,kegg_id,Reactants,Products,EC,GPR,b_number,uniprot_id,gene_id
0,12DGR120tipp,NaN,[],[],NaN,,NaN,NaN,NaN
1,12DGR140tipp,NaN,[],[],NaN,,NaN,NaN,NaN
2,12DGR161tipp,NaN,[],[],NaN,,NaN,NaN,NaN
3,12DGR180tipp,NaN,[],[],NaN,,NaN,NaN,NaN
4,12DGR181tipp,NaN,[],[],NaN,,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3057,GLYCLTt2rpp,NaN,"[C00160, C00080]","[C00160, C00080]",NaN,,NaN,NaN,NaN
3058,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,NaN,WP_011014733_1
3059,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,NaN,WP_011014733_1
3060,CYTB1,NaN,[],[],NaN,,NaN,NaN,NaN


In [8]:
def extract_kegg_gene_ids(text):
    if pd.isna(text):
        return text
    return re.findall(r'Cgl\d{4}', text)

# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['b_number'] = uniprot_df['Gene Names'].apply(extract_b_numbers)
# uniprot_df['kegg_gene'] = uniprot_df['Gene Names'].apply(extract_kegg_gene_ids)
uniprot_df = uniprot_df.explode('b_number', ignore_index=True)
# uniprot_df = uniprot_df.explode('kegg_gene', ignore_index=True)

uniprot_df = uniprot_df.drop(['Gene Names'], axis=1)
uniprot_df_genemapper = uniprot_df[['Entry', 'Mass', 'Length']]


### 2.3 Match Uniprot to BIGG data

In [9]:
# Merge first on b_number
id_mapper_with_protein = pd.merge(
    id_mapper_df, 
    uniprot_df_genemapper, 
    left_on='uniprot_id', 
    right_on = 'Entry',
    how='left'
)

id_mapper_with_protein = id_mapper_with_protein.drop('Entry', axis=1)
id_mapper_with_protein

,rxn_id,kegg_id,Reactants,Products,EC,GPR,b_number,uniprot_id,gene_id,Mass,Length
0,12DGR120tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN
1,12DGR140tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN
2,12DGR161tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN
3,12DGR180tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN
4,12DGR181tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3545,GLYCLTt2rpp,NaN,"[C00160, C00080]","[C00160, C00080]",NaN,,NaN,NaN,NaN,NaN,NaN
3546,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,NaN,WP_011014733_1,NaN,NaN
3547,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,NaN,WP_011014733_1,NaN,NaN
3548,CYTB1,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN


## 3. Match the BiGG model reaction and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [10]:
id_mapper_final = id_mapper_with_protein.copy()
id_mapper_final = id_mapper_final.rename({'uniprot_id':'enzyme_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg_id', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

,rxn_id,kegg_id,Reactants,Products,EC,GPR,b_number,enzyme_id,gene_id,Mass,Length
0,12DGR120tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN
1,12DGR140tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN
2,12DGR161tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN
3,12DGR180tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN
4,12DGR181tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN


In [11]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,Cgl0023,R01664,3.1.3.5,C00239,5.4287
1,Cgl0023,R01664,3.1.3.89,C00239,5.4287
2,Cgl0023,R01664,3.1.3.-,C00239,5.4287
3,Cgl0023,R00511,3.1.3.5,C00475,5.9248
4,Cgl0023,R00511,3.1.3.91,C00475,5.9248


In [12]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = id_mapper_final.merge(eco_enzymes_df, 
                              left_on =['b_number', 'kegg_id'], 
                              right_on = ['gene', 'reaction_id'],
                             how = 'left').drop(['gene', 'reaction_id'], axis=1).rename({'b_number':'gene'}, axis=1)
eco_enzymes_merged

,rxn_id,kegg_id,Reactants,Products,EC,GPR,gene,enzyme_id,gene_id,Mass,Length,ec_number,compound,kcat_values
0,12DGR120tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12DGR140tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12DGR161tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12DGR180tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12DGR181tipp,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3633,GLYCLTt2rpp,NaN,"[C00160, C00080]","[C00160, C00080]",NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3634,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,NaN,WP_011014733_1,NaN,NaN,NaN,NaN,NaN
3635,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,NaN,WP_011014733_1,NaN,NaN,NaN,NaN,NaN
3636,CYTB1,NaN,[],[],NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [13]:
def assign_missing_gprs(df: pd.DataFrame, use_ec: bool = False) -> pd.DataFrame:
    """Assigns default gene, GPR, and enzyme IDs for unmappable proteins.

    This function updates the 'gene', 'GPR', and 'enzyme_id' columns for rows where 
    'GPR' is missing (empty string). If `use_ec=True`, EC numbers are also used 
    to generate gene and enzyme identifiers.

    Args:
        df (pd.DataFrame): A DataFrame containing enzyme reaction data, 
            including 'GPR', 'rxn_id', 'gene', 'enzyme_id', and optionally 'EC'.
        use_ec (bool, optional): Whether to include EC numbers in the assigned values. 
            Defaults to False.

    Returns:
        pd.DataFrame: The updated DataFrame with missing 'gene', 'GPR', and 'enzyme_id' 
        values assigned where applicable.
    """
    # Ensure GPR is treated as a string and replace NaNs with empty strings
    df['GPR'] = df['GPR'].fillna('')
    
    # Condition: GPR is missing
    missing_gene_mask = (df['GPR'] == '')

    if use_ec:
        missing_gene_mask &= df['EC'].notna()
        id_pattern = lambda row: (f'Gene_{row.rxn_id}_{row.EC}', 
                                  f'Gene_{row.rxn_id}_{row.EC}', 
                                  f'Enzyme_{row.rxn_id}_{row.EC}')
    else:
        id_pattern = lambda row: (f'Gene_{row.rxn_id}', 
                                  f'Gene_{row.rxn_id}', 
                                  f'Enzyme_{row.rxn_id}')
    
    # Apply only to rows that meet the condition
    new_values = df.loc[missing_gene_mask].apply(lambda row: pd.Series(id_pattern(row)), axis=1)
    
    # Assign values
    df.loc[missing_gene_mask, ['gene', 'GPR', 'enzyme_id']] = new_values.to_numpy()
    
    return df

# Set default values
eco_enzymes_parsed = eco_enzymes_merged.copy()
eco_enzymes_parsed['direction'] = 'f'
eco_enzymes_parsed['kcat_values'] = eco_enzymes_parsed['kcat_values'].fillna(DEFAULT_KCAT)
eco_enzymes_parsed['Mass'] = eco_enzymes_parsed['Mass'].fillna(DEFAULT_PROT_MW)
eco_enzymes_parsed['Length'] = eco_enzymes_parsed['Length'].fillna(DEFAULT_PROT_LENGTH)



# Determine direction of the reaction by determining whether the kcat is assigned to the enzyme-substrate or enzyme-product pain
eco_enzymes_parsed['direction'] = np.where(
    eco_enzymes_parsed['compound'].isin(eco_enzymes_parsed['Products']), 'b',
    np.where(eco_enzymes_parsed['compound'].isin(eco_enzymes_parsed['Reactants']), 
             'f', 
             eco_enzymes_parsed['direction'])
)

# Assign default genes for unmappable proteins
#if there is an ec number available
eco_enzymes_parsed = assign_missing_gprs(eco_enzymes_parsed, use_ec =True)
#if there is no information available
eco_enzymes_parsed = assign_missing_gprs(eco_enzymes_parsed, use_ec = False)

# Assign enzyme IDs if missing but gene information is available
eco_enzymes_parsed['enzyme_id'] = np.where(
    eco_enzymes_parsed['enzyme_id'].isna(),
    eco_enzymes_parsed['gene'].apply(lambda x: f'Enzyme_{x}' if isinstance(x, str) else None),
    eco_enzymes_parsed['enzyme_id']
)

# Assign genes if missing if enzyme information is available
eco_enzymes_parsed['gene'] = eco_enzymes_parsed.apply(
    lambda row: f'gene_{row.enzyme_id}' if isinstance(row.gene, float) else row.gene, axis=1
)

# Handle specific case for 's0001' (annotation is gene is not mapped to a genome annotation)
eco_enzymes_parsed.loc[eco_enzymes_parsed['GPR'] == 's0001', 'gene'] = eco_enzymes_parsed['GPR']
eco_enzymes_parsed.loc[eco_enzymes_parsed['GPR'] == 's0001', 'enzyme_id'] = 'Enzyme_s0001' + eco_enzymes_parsed['rxn_id']


# Clean up final columns
eco_enzymes_parsed = eco_enzymes_parsed.drop(['ec_number', 'compound', 'gene_id'], axis=1).rename(columns={'Mass': 'molMass'})
eco_enzymes_parsed

,rxn_id,kegg_id,Reactants,Products,EC,GPR,gene,enzyme_id,molMass,Length,kcat_values,direction
0,12DGR120tipp,NaN,[],[],NaN,Gene_12DGR120tipp,Gene_12DGR120tipp,Enzyme_12DGR120tipp,39959.4825,325.0,13.7,f
1,12DGR140tipp,NaN,[],[],NaN,Gene_12DGR140tipp,Gene_12DGR140tipp,Enzyme_12DGR140tipp,39959.4825,325.0,13.7,f
2,12DGR161tipp,NaN,[],[],NaN,Gene_12DGR161tipp,Gene_12DGR161tipp,Enzyme_12DGR161tipp,39959.4825,325.0,13.7,f
3,12DGR180tipp,NaN,[],[],NaN,Gene_12DGR180tipp,Gene_12DGR180tipp,Enzyme_12DGR180tipp,39959.4825,325.0,13.7,f
4,12DGR181tipp,NaN,[],[],NaN,Gene_12DGR181tipp,Gene_12DGR181tipp,Enzyme_12DGR181tipp,39959.4825,325.0,13.7,f
...,...,...,...,...,...,...,...,...,...,...,...,...
3633,GLYCLTt2rpp,NaN,"[C00160, C00080]","[C00160, C00080]",NaN,Gene_GLYCLTt2rpp,Gene_GLYCLTt2rpp,Enzyme_GLYCLTt2rpp,39959.4825,325.0,13.7,f
3634,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,Enzyme_WP_011014733_1,39959.4825,325.0,13.7,f
3635,PYRt,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,Enzyme_WP_011014733_1,39959.4825,325.0,13.7,f
3636,CYTB1,NaN,[],[],NaN,Gene_CYTB1,Gene_CYTB1,Enzyme_CYTB1,39959.4825,325.0,13.7,f


In [14]:
eco_enzymes_mapped = eco_enzymes_parsed.copy()
protein2gene, gene2protein = get_protein_gene_mapping(eco_enzymes_mapped, model)
# eco_enzymes_mapped.gene = [[g] for g in eco_enzymes_mapped.gene]

gene2protein = {replace_ids(k, gene_mapper): v for k, v in gene2protein.items()}

# Ensure the enzyme complexes are merged on one row
eco_enzymes_mapped = merge_enzyme_complexes(eco_enzymes_mapped, gene2protein)

# Save the adjusted dataframe or continue processing
eco_enzymes_mapped

KeyboardInterrupt: 

In [ ]:
#save the dataframe
#drop duplicate entries. If there are duplicates, make sure the mean kcat value is used to parametrize
eco_enzymes_final = eco_enzymes_mapped.groupby(
    ['rxn_id', 'enzyme_id', 'direction'], 
    as_index=False
).agg({
    'kcat_values': 'mean', 
    **{col: 'first' for col in eco_enzymes_mapped.columns if col not in ['rxn_id', 'enzyme_id', 'direction', 'kcat']
      }
})
# eco_enzymes_final.to_excel(AE_OUTPUT_FILE_PATH)

### 7. Save the dataframe to the proper format for building PAMs

In [32]:
# Get the information about the enzyme sectors
other_sectors = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = None)
del other_sectors['ActiveEnzymes']

# adjust unused enzymes information
unused_enzyme_information = other_sectors['ExcessEnzymes'].set_index('Parameter')
unused_enzyme_information.loc['id_list', 'Value'] = BIOMASS_RXN_ID
unused_enzyme_information.loc['ups_mu', 'Value'] = -UNUSED_PROTEIN_INTERCEPT*METABOLIC_PROTEIN_FRACTION/MAX_GROWTH_ALE
unused_enzyme_information.loc['ups_0', 'Value'] = UNUSED_PROTEIN_INTERCEPT*METABOLIC_PROTEIN_FRACTION

del other_sectors['ExcessEnzymes']
other_sectors['UnusedEnzyme'] = unused_enzyme_information.reset_index()

In [79]:
# Save it to a new excel file
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_final.to_excel(writer, sheet_name='ActiveEnzymes', index = False)
    for sheet, df in other_sectors.items():
        if sheet == 'ExcessEnzymes': sheet = 'UnusedEnzyme'
        df.to_excel(writer, sheet_name=sheet, index = False)

In [33]:
other_sectors['UnusedEnzyme']

,Parameter,Value,Unit,Description
0,id_list,Growth,NaN,biomass formation reaction ID (representing gr...
1,ups_mu,-0.096354,g/g_CDW/h,slope of relation between growth rate and prot...
2,ups_0,0.0925,g/g_CDW,Protein concentration allocated to excess enzy...
3,mol_mass,39477.787843,g/mol,Molar mass of a fictional enzyme
